## 1. Importar librerías necesarias
Importamos las librerías esenciales para conectar a las bases de datos PostgreSQL, manipular datos con pandas y leer configuración desde YAML.

In [8]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

## 2. Cargar configuración y crear conexiones
Leemos el archivo `config.yml` para obtener las credenciales de ambas bases de datos (origen MENSAJERIA_OLTP y destino ETL_PROCESS). Creamos dos motores SQLAlchemy para conectarnos a cada una.

In [9]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_mensajeria = config['MENSAJERIA_OLTP']
    config_etl = config['ETL_PROCESS']

# Construct the database URL
url_mensajeria = (f"{config_mensajeria['drivername']}://{config_mensajeria['user']}:{config_mensajeria['password']}@{config_mensajeria['host']}:"
          f"{config_mensajeria['port']}/{config_mensajeria['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")

# Create the SQLAlchemy Engine
mensajeria = create_engine(url_mensajeria)
etl_conn = create_engine(url_etl)

## 3. Extraer y fusionar datos de mensajeros
Leemos las tablas `clientes_mensajeroaquitoy` y `auth_user` de la base de datos OLTP. Realizamos un merge (join) usando `user_id` para combinar la información de mensajeros con sus nombres de usuario. Seleccionamos solo las columnas necesarias.

In [10]:
mensajeroAquiToy = pd.read_sql_table('clientes_mensajeroaquitoy', mensajeria)
auth_user = pd.read_sql_table('auth_user', mensajeria)

mensajeroAquiToy

auth_user = auth_user[['id', 'username']]
auth_user

dim_mensajero = mensajeroAquiToy.merge(
    auth_user,
    left_on='user_id',
    right_on='id',
    how='left'
)

dim_mensajero = dim_mensajero.rename(columns={'id_x': 'mensajero_id', 'username': 'nombre_completo'})
dim_mensajero = dim_mensajero[['mensajero_id', 'nombre_completo']]

dim_mensajero

,mensajero_id,nombre_completo
0,1,mensajero1
1,42,JPEDROZA
2,48,JULIANVILLANUEVA
3,41,LUISCARDONA
4,13,GEOVANNY Hidalgo
5,28,JHONMUÑOZ
6,33,JONATANMANZANO
7,36,LUISGIL
8,8,Luis Castro
9,21,Vladimir Putin


## 4. Limpiar y transformar datos
Eliminamos duplicados, ordenamos por ID, reseteamos índices y creamos una nueva columna `mensajero_key` con valores secuenciales. Rellenamos valores nulos en nombres con "Sin nombre". Este es el resultado final de la dimensión.

In [11]:
dim_mensajero = (
    dim_mensajero
    .drop_duplicates(subset=['mensajero_id'])
    .sort_values('mensajero_id')
    .reset_index(drop=True)
)

dim_mensajero['mensajero_key'] = dim_mensajero.index + 1

dim_mensajero['nombre_completo'] = dim_mensajero['nombre_completo'].fillna('Sin nombre')

dim_mensajero = dim_mensajero[['mensajero_key', 'mensajero_id', 'nombre_completo']]

dim_mensajero

,mensajero_key,mensajero_id,nombre_completo
0,1,1,mensajero1
1,2,2,mensajero2
2,3,3,Biil-Gates
3,4,4,Lionel_messi
4,5,5,James Rodriguez
5,6,6,mensajero_o
6,7,7,admin
7,8,8,Luis Castro
8,9,9,DIEGO Maradona
9,10,10,Luis Valenciano


In [12]:
fila_no_aplica = pd.DataFrame(
    [{"mensajero_key": -1, "mensajero_id": -1, "nombre_completo": "No aplica"}]
)

dim_mensajero = pd.concat([dim_mensajero, fila_no_aplica], ignore_index=True)

## 5. Limpiar tabla destino e insertar datos
Primero, vaciamos la tabla `dim_mensajero` en la base de datos ETL (truncate). Luego, insertamos el dataframe limpio y transformado en esa tabla. Esto asegura que los datos estén actualizados sin duplicados.

In [13]:
#with etl_conn.connect() as conn:
#    conn.execute(db.text("TRUNCATE TABLE dim_mensajero RESTART IDENTITY"))
#    conn.commit()
dim_mensajero.to_sql('dim_mensajero', etl_conn, if_exists='replace', index=False)

dim_mensajero

,mensajero_key,mensajero_id,nombre_completo
0,1,1,mensajero1
1,2,2,mensajero2
2,3,3,Biil-Gates
3,4,4,Lionel_messi
4,5,5,James Rodriguez
5,6,6,mensajero_o
6,7,7,admin
7,8,8,Luis Castro
8,9,9,DIEGO Maradona
9,10,10,Luis Valenciano


## 6. Verificar datos insertados
Ejecutamos una consulta SELECT para confirmar que los datos fueron insertados correctamente en la base de datos ETL.

In [14]:
pd.read_sql('SELECT * FROM dim_mensajero', etl_conn)

,mensajero_key,mensajero_id,nombre_completo
0,1,1,mensajero1
1,2,2,mensajero2
2,3,3,Biil-Gates
3,4,4,Lionel_messi
4,5,5,James Rodriguez
5,6,6,mensajero_o
6,7,7,admin
7,8,8,Luis Castro
8,9,9,DIEGO Maradona
9,10,10,Luis Valenciano
